# Real-Space Lattices in TightBinding

**Abstract.** This notebook introduces the real-space lattice layer of `tightbinding_py`, the faithful Python port of `TightBinding.jl`. We build a lattice from its Bravais vectors and sublattice positions in *crystal coordinates*, enumerate the finite sample of unit cells and sites, compute the primitive-cell volume, enforce periodic boundary conditions through a minimum-image convention, and inspect the nearest-neighbour graph built by Euclidean distance. We close with twisted boundary phases (Aharonov–Bohm fluxes), which drive the band-topology and many-body-Chern notebooks.

**References**

- N. W. Ashcroft and N. D. Mermin, *Solid State Physics* (Saunders College Publishing, 1976), Chs. 4–5.
- D. N. Sheng et al., Phys. Rev. Lett. **107**, 146803 (2011).
- K. Sun, Z. Gu, H. Katsura, and S. Das Sarma, Phys. Rev. Lett. **106**, 236803 (2011).
- T. Fukui, Y. Hatsugai, and H. Suzuki, J. Phys. Soc. Jpn. **74**, 1674 (2005).
- R. Resta, Rev. Mod. Phys. **66**, 899 (1994).


## 1. Bravais vectors and crystal coordinates

A $D$-dimensional Bravais lattice is the set of points

\begin{equation}
\mathbf R_{\mathbf n} = \sum_{d=1}^{D} n_d\,\mathbf a_d,
\qquad n_d \in \mathbb Z,
\end{equation}

generated by $D$ linearly independent **Bravais vectors** $\{\mathbf a_d\}$. The integers $\mathbf n = (n_1,\dots,n_D)$ are the **crystal coordinates** of the lattice point. When a lattice has more than one atom per unit cell (a *lattice with a basis*), each atom type $\alpha$ sits at a fixed offset $\boldsymbol\tau_\alpha$ inside the unit cell, also given in crystal coordinates, so that

\begin{equation}
\mathbf r_{\mathbf n,\alpha}
= \sum_{d} \bigl(n_d + (\tau_\alpha)_d\bigr)\,\mathbf a_d .
\end{equation}

The package stores the Bravais vectors as the rows of a matrix $A$ (`brav_vec_list`). A row vector of crystal coordinates $\mathbf c$ maps to Cartesian coordinates by right multiplication,

\begin{equation}
\mathbf r_{\mathrm{cart}} = \mathbf c \cdot A
\qquad\text{(i.e.}\ \mathbf r_{\mathrm{cart}} = \sum_d c_d\,\mathbf a_d\text{)},
\end{equation}

and the **primitive-cell volume** is the unsigned determinant

\begin{equation}
\Omega = |\det A| .
\end{equation}

> **Remark.** The sublattice offsets $\boldsymbol\tau_\alpha$ are stored *in crystal coordinates* (`sub_crys_list`), keeping the geometric (Bravais) and combinatorial (sublattice) information cleanly separated.


## 2. Building lattices with `initialize_real_space_lattice`

The port exposes one constructor, `initialize_real_space_lattice`, with keyword arguments:

```python
initialize_real_space_lattice(
    lattice_name="honeycomb",     # exact preset name (overrides the three below), or ""
    brav_vec_list=[[a1x, a1y], [a2x, a2y]],
    sub_crys_list=[[tau1x, tau1y], [tau2x, tau2y]],
    sample_size=[L1, L2],         # unit cells along each Bravais direction
    pbc_indicator=[True, True],   # periodic boundary conditions per direction
    twisted_phases_over_2π=[0.0, 0.0],
    allowed_bonds=None,           # optional 1-based sublattice-pair filter for the graph
)
```

The presets (exact match, Julia `@match` order) are `"square"`, `"honeycomb"`, `"kagome"`, `"Lieb"`, `"dice"` — note the capital **L** in `"Lieb"`. There is no `"checkerboard"` preset; it is built manually below. The finite sample has

\begin{equation}
n_{\mathrm{cell}} = \prod_d L_d,
\qquad\qquad
n_{\mathrm{site}} = n_{\mathrm{cell}}\cdot n_{\mathrm{sub}} .
\end{equation}

All site, cell and sublattice indices in the package are **1-based**, exactly as in the Julia source.


In [ ]:
import numpy as np
from tightbinding_py import *

# The five built-in presets plus a manually-built checkerboard lattice.
def show(name, lattice):
    print(f"{name:12s} n_sub={lattice.n_sub}  n_cell={lattice.n_cell}  n_site={lattice.n_site}"
          f"  cell_volume={lattice.cell_volume:.6f}  graph_edges={lattice.graph.n_edges()}"
          f"  subs={lattice.sub_name_list}")

for nm in ["square", "honeycomb", "kagome", "Lieb", "dice"]:
    lat = initialize_real_space_lattice(lattice_name=nm, sample_size=[3, 3],
                                        pbc_indicator=[True, True])
    show(nm, lat)

# Checkerboard: square Bravais, two sublattices at (0,0) and (1/2,1/2).
checkerboard = initialize_real_space_lattice(
    brav_vec_list=[[1.0, 0.0], [0.0, 1.0]],
    sample_size=[3, 3],
    sub_crys_list=[[0.0, 0.0], [0.5, 0.5]],
    lattice_name="checkerboard",
    pbc_indicator=[True, True],
)
show("checkerboard", checkerboard)


**Check the numbers.** For honeycomb on a $3\times 3$ sample, $n_{\mathrm{cell}} = 9$, $n_{\mathrm{sub}} = 2$, so $n_{\mathrm{site}} = 18$; the triangular Bravais basis $\mathbf a_1=(1,0)$, $\mathbf a_2=(1/2,\sqrt3/2)$ gives $\Omega = \sqrt3/2 \approx 0.866$. Kagome and dice share that basis ($\Omega=0.866$), while square and Lieb use $\mathbf a_1=(1,0)$, $\mathbf a_2=(0,1)$ with $\Omega=1$. The graph edge counts are the *undirected* nearest-neighbour bonds: honeycomb $18\times3/2=27$, square $9\times4/2=18$, kagome $27\times4/2=54$, Lieb $36$, dice (hub–rim) $54$, checkerboard $36$.


## 3. Site enumeration and index maps

The sample is enumerated by a direct product of integer ranges in Julia's `Iterators.product` order (first axis fastest), so `cell_int_list` is a list of integer *tuples*, and

```python
site_list = [(cell_int, i_sub) for cell_int in cell_int_list for i_sub in 1:n_sub]
```

A site is a `(cell_int, i_sub)` pair with `i_sub` **1-based** (e.g. `((2, 1), 1)` is sublattice 1 of cell `(2, 1)`). The linear index is `lattice.site_to_index_map[(cell_int, i_sub)]`, also 1-based. The crystal and Cartesian positions of every site are precomputed as the lists `site_crys_list` and `site_cart_list` (entry $i-1$ belongs to the $i$-th site of `site_list`).


In [ ]:
lat = initialize_real_space_lattice(lattice_name="honeycomb", sample_size=[2, 2],
                                       pbc_indicator=[True, True])

print("cell_int_list =", lat.cell_int_list)
print("n_cell =", lat.n_cell, "   n_site =", lat.n_site)
print("sub_name_list =", lat.sub_name_list)
print("site_list[:6] =", lat.site_list[:6])
print()
print("site_crys_list[:4] =", [np.round(c, 6).tolist() for c in lat.site_crys_list[:4]])
print("site_cart_list[:4] =", [np.round(c, 6).tolist() for c in lat.site_cart_list[:4]])
print()
print("site_to_index_map[((1, 0), 2)] =", lat.site_to_index_map[((1, 0), 2)])
print("site_to_index_map[((0, 0), 1)] =", lat.site_to_index_map[((0, 0), 1)])


**Ordering.** `site_list` runs the sublattice index fastest (all `n_sub` sites of cell `(0,0)` first, then cell `(1,0)`, …). This lexicographic (cell-major) order is used throughout the package and is what makes the many-body Slater determinant block cleanly in the flux-torus notebook.


## 4. The six lattices

| name | Bravais vectors | sublattice offsets $\boldsymbol\tau_\alpha$ (crystal) | $n_{\mathrm{sub}}$ | graph |
|---|---|---|---|---|
| `square` | $(1,0),\,(0,1)$ | $(0,0)$ | 1 | all pairs |
| `honeycomb` | $(1,0),\,(\tfrac12,\tfrac{\sqrt3}{2})$ | $(0,0),\,(\tfrac13,\tfrac13)$ | 2 | all pairs |
| `kagome` | $(1,0),\,(\tfrac12,\tfrac{\sqrt3}{2})$ | $(0,0),\,(\tfrac12,0),\,(0,\tfrac12)$ | 3 | all pairs |
| `Lieb` | $(1,0),\,(0,1)$ | $(0,0),\,(\tfrac12,0),\,(0,\tfrac12)$ | 3 | all pairs |
| `checkerboard` (manual) | $(1,0),\,(0,1)$ | $(0,0),\,(\tfrac12,\tfrac12)$ | 2 | all pairs |
| `dice` (T3) | $(1,0),\,(\tfrac12,\tfrac{\sqrt3}{2})$ | $(0,0),\,(\tfrac13,\tfrac13),\,(\tfrac23,\tfrac23)$ | 3 | `allowed_bonds=[(1,2),(2,3)]` |

The `dice` preset restricts the graph to the sublattice pairs $(1,2)$ and $(2,3)$, so sublattice `A2` becomes the hub of the $\alpha$-T3 lattice, connected to three `A1` and three `A3` rim sites.


## 5. Periodic boundary conditions and the minimum-image convention

For a finite sample of size $(L_1,\dots,L_D)$, a bond template displacing a site by $\Delta\mathbf c$ connects $\mathbf c$ to $\mathbf c + \Delta\mathbf c$. In a direction $d$ where `pbc_indicator[d]` is `True`, the target cell is wrapped back into the sample,

\begin{equation}
(\mathbf c + \Delta\mathbf c)_d \mapsto \bigl[(\mathbf c + \Delta\mathbf c)_d \bmod L_d\bigr].
\end{equation}

The **minimum-image convention** wraps a displacement into the range $[-L_d/2,\, L_d/2]$ along each periodic direction before its Euclidean length is compared. This is exactly how the nearest-neighbour graph is built: every site pair is measured through its *shortest* periodic image, so a bond that would cross the boundary is connected across the sample instead of being dropped (and is drawn with a "ghost" partner in plots). In an **open** direction the minimum image is *not* applied — the graph simply lacks the bonds that would leave the sample.


In [ ]:
# Torus vs cylinder: opening one direction removes its wrapping bonds.
torus = initialize_real_space_lattice(lattice_name="honeycomb", sample_size=[3, 2],
                                      pbc_indicator=[True, True])
cylinder = initialize_real_space_lattice(lattice_name="honeycomb", sample_size=[3, 2],
                                         pbc_indicator=[True, False])
print("torus    (PBC in both directions) :", torus.graph.n_edges(), "undirected NN edges")
print("cylinder (PBC in x, open in y)    :", cylinder.graph.n_edges(), "undirected NN edges")

# A wrapped bond appears in the graph as an edge between far-apart sites.
# For a [1,1] torus the two sites of the single cell are linked by all three
# periodic images collapsing onto one edge:
tiny = initialize_real_space_lattice(lattice_name="honeycomb", sample_size=[1, 1],
                                     pbc_indicator=[True, True])
print("1x1 torus edges:", tiny.graph.edges())


**Reading the output.** On the torus the $x$- and $y$-boundary bonds are present; on the cylinder the bonds crossing the open $y$-direction are gone, so the edge count drops. The `[1,1]` example shows the extreme case: the two sites of the single unit cell are nearest neighbours through *every* periodic image, all collapsing onto the single edge `(1, 2)`.


## 6. The nearest-neighbour graph (`LatticeGraph`)

The graph is a thin adapter (`LatticeGraph`) over an `igraph.Graph`, exposing the subset of the Julia `Graphs.SimpleGraph` API used by the package (all **1-based**):

- `graph.neighbors(i)` — set of neighbours of site `i`;
- `graph.gdistances(i)` — dict `site → graph distance` from site `i`;
- `graph.edges()` — list of undirected edges `(i, j)` with `i < j`;
- `graph.n_edges()` — edge count;
- `graph.igraph_graph` — the underlying igraph object (0-based vertex ids, vertex attribute `"site"`).

The graph is built by comparing minimal Euclidean distances with the PBC minimum-image convention, restricted to the `allowed_bonds` sublattice pairs when provided.


In [ ]:
lat = initialize_real_space_lattice(lattice_name="honeycomb", sample_size=[3, 3],
                                       pbc_indicator=[True, True])
g = lat.graph
print("neighbors(1)  =", g.neighbors(1))
print("gdistances(1) =", g.gdistances(1))
print("n_edges       =", g.n_edges())
print("first 5 edges =", g.edges()[:5])

ig = g.igraph_graph
print("\nigraph: vertices =", ig.vcount(), " edges =", ig.ecount())
print("igraph vertex 0 site:", ig.vs[0]["site"])


## 7. Plotting the lattice

`plot_real_space_lattice(lattice, save_path=...)` draws the sites (coloured by sublattice), the bonds, a dashed unit-cell grid, the Bravais arrows $\mathbf a_1,\mathbf a_2$, and — for periodic samples — transparent **ghost sites** where a wrapped bond exits the sample. The cells below write SVGs into `doc/figures/`.

> **Execution note.** This notebook was authored in an environment without `nbconvert`, so its cells ship *unexecuted but correct*; the SVGs referenced below were produced by running the identical code out-of-band with the package venv. Run the cells yourself (from the package root `TightBinding_PY/`, so that `doc/figures/` resolves) to regenerate them.


In [ ]:
from pathlib import Path

FIG_DIR = Path("doc") / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

lat_hc = initialize_real_space_lattice(lattice_name="honeycomb", sample_size=[3, 3],
                                       pbc_indicator=[True, True])
plot_real_space_lattice(lat_hc, save_path=FIG_DIR / "lattice_honeycomb.svg")

lat_kg = initialize_real_space_lattice(lattice_name="kagome", sample_size=[3, 3],
                                       pbc_indicator=[True, True])
plot_real_space_lattice(lat_kg, save_path=FIG_DIR / "lattice_kagome.svg")


## 8. Twisted boundary phases (Aharonov–Bohm fluxes)

Threading a flux $\Phi_d = 2\pi\theta_d$ through a periodic direction $d$ modifies the boundary condition to

\begin{equation}
\psi(\dots, x_d + L_d, \dots) = e^{i2\pi\theta_d}\,\psi(\dots, x_d, \dots),
\qquad \theta_d = \frac{\Phi_d}{2\pi}.
\end{equation}

The lattice stores the dimensionless twists in `twisted_phases_over_2π` (default zeros), and construction *enforces* that a non-zero twist is only allowed along a periodic direction. Two equivalent viewpoints recur throughout the package:

1. **Flux = momentum shift.** On the momentum grid the twist shifts the crystal momenta, $k_d = (n_d + \theta_d)/L_d$.
2. **Flux = twisted real-space Hamiltonian.** The real-space Hamiltonian built with twist $\boldsymbol\theta$ (via `build_real_space_tb_Hamiltonain`) is the object whose spectrum is pumped across the flux torus — the many-body Chern number of the last notebook.


In [ ]:
# A torus threaded by theta_1 = 0.6 (i.e. flux 0.6 * 2pi through direction 1).
lat_flux = initialize_real_space_lattice(
    lattice_name="honeycomb", sample_size=[3, 3],
    pbc_indicator=[True, True], twisted_phases_over_2π=[0.6, 0.0],
)
print("stored twisted_phases_over_2π =", lat_flux.twisted_phases_over_2π)

# The momentum grid is automatically shifted: k_1 = (n_1 + 0.6) / 3.
k_grid = initialize_uniform_grids_from_lattice(lat_flux)
print("first k-point (crystal):", np.round(k_grid.site_crys_list[0], 6).tolist())
print("sample_size:", k_grid.sample_size, "  nsite:", k_grid.nsite)

# Validation: a twist on an OPEN direction is rejected.
try:
    initialize_real_space_lattice(lattice_name="honeycomb", sample_size=[3, 3],
                                  pbc_indicator=[True, False],
                                  twisted_phases_over_2π=[0.0, 0.5])
except ValueError as e:
    print("ValueError:", str(e)[:120], "...")


## 9. Summary and finite-size effects

- `initialize_real_space_lattice` bundles Bravais vectors, sublattice offsets, a finite cell/site enumeration, and a Euclidean-distance nearest-neighbour graph.
- Crystal coordinates map to Cartesian by `crys @ brav_vec_list`; the cell volume is $|\det A|$.
- PBC wrapping realises the minimum-image convention; open directions drop boundary bonds.
- Twisted phases encode flux threading; they shift the momentum grid and will drive the topological response in later notebooks.

**Finite-size effects to keep in mind.** The site count grows as $n_{\mathrm{site}} = n_{\mathrm{sub}}\prod_d L_d$, and the $O(n_{\mathrm{site}}^2)$ neighbour search becomes costly quickly. For topological quantisation we will usually *increase the momentum-grid resolution* (not the real-space sample) to converge integer invariants, while keeping the real-space sample small enough for exact diagonalisation — the tension between these two is a recurring theme of the remaining notebooks.
